In [ ]:
from cloudevents.http import CloudEvent

import functions_framework


# Triggered by a change in a storage bucket
@functions_framework.cloud_event
def hello_gcs(cloud_event: CloudEvent) -> tuple:
    """This function is triggered by a change in a storage bucket.

    Args:
        cloud_event: The CloudEvent that triggered this function.
    Returns:
        The event ID, event type, bucket, name, metageneration, and timeCreated.
    """
    data = cloud_event.data

    event_id = cloud_event["id"]
    event_type = cloud_event["type"]

    bucket = data["bucket"]
    name = data["name"]
    metageneration = data["metageneration"]
    timeCreated = data["timeCreated"]
    updated = data["updated"]

    print(f"Event ID: {event_id}")
    print(f"Event type: {event_type}")
    print(f"Bucket: {bucket}")
    print(f"File: {name}")
    print(f"Metageneration: {metageneration}")
    print(f"Created: {timeCreated}")
    print(f"Updated: {updated}")

    return event_id, event_type, bucket, name, metageneration, timeCreated, updated


In [ ]:
# 화자분할 기능 테스트

import os
# Google Cloud 인증 설정 - 로컬 개발 환경에서 사용
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/hyunjunson/study/python/nipa-oracle/gaon-service-account.json"

from google.cloud import speech

client = speech.SpeechClient()

# GCS URI 사용
audio = speech.RecognitionAudio(uri="gs://gaon-cloud-data/sound-raw-data/sample/1040.mp3")

# 발화자 구분 옵션
diarization_config = speech.SpeakerDiarizationConfig(
    enable_speaker_diarization=True,
    min_speaker_count=2,
    max_speaker_count=3,
)

config = speech.RecognitionConfig(
    encoding=speech.RecognitionConfig.AudioEncoding.MP3,  # MP3 파일일 경우
    sample_rate_hertz=44100, # 보통 mp3는 44100
    language_code="ko-KR", # 한국어 코드
    diarization_config=diarization_config,
    enable_automatic_punctuation=True,  # 문장부호 자동추가
    enable_separate_recognition_per_channel=True,
    audio_channel_count=2
)

print("Waiting for operation to complete...")
# 비동기 인식 요청
operation = client.long_running_recognize(config=config, audio=audio)

# 작업 완료 대기
response = operation.result(timeout=300)  # 최대 5분 대기

# The transcript within each result is separate and sequential per result.
# However, the words list within an alternative includes all the words
# from all the results thus far. Thus, to get all the words with speaker
# tags, you only have to take the words list from the last result:
result = response.results[-1]

words_info = result.alternatives[0].words
version = 3
# Printing out the output
# txt 파일로 output 저장
output_file = f"transcription_output_ver{version}.txt"

with open(output_file, 'w', encoding='utf-8') as f:
    for word_info in words_info:
        f.write(f"word: '{word_info.word}', speaker_tag: {word_info.speaker_tag}\n")

print(f"Transcription saved to {output_file}")

E0000 00:00:1762310665.590701  576705 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


Waiting for operation to complete...
Transcription saved to transcription_output_ver2.txt


In [2]:
from google.cloud import speech


def transcribe_diarization_gcs_beta(audio_uri: str) -> bool:
    """Google Cloud Storage에 저장된 원격 오디오 파일을 화자 분할(Speaker Diarization) 기능을 사용해
    텍스트로 변환합니다.
    Args:
        audio_uri (str): 오디오 파일의 GCS 경로 (예: gs://[BUCKET]/[FILE])
    Returns:
        변환이 성공적으로 완료되면 True, 그렇지 않으면 False를 반환합니다.
    """

    client = speech.SpeechClient()

    # 화자 분할 설정 (최소/최대 화자 수 조정)
    speaker_diarization_config = speech.SpeakerDiarizationConfig(
        enable_speaker_diarization=True,
        min_speaker_count=2,  # 최소 화자 수
        max_speaker_count=3,  # 예상되는 최대 화자 수
    )

    # 음성 인식 구성 (오디오 인코딩, 언어, 샘플레이트 등)
    recognition_config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.MP3,
        language_code="ko-KR",
        sample_rate_hertz=44100,
        diarization_config=speaker_diarization_config,
        enable_automatic_punctuation=True
    )


    # 원격 오디오 파일 경로 설정
    audio = speech.RecognitionAudio(
        uri="gs://gaon-cloud-data/sound-raw-data/sample/1040.mp3"
    )

    # 비동기 방식으로 긴 오디오 파일을 변환 요청
    response = client.long_running_recognize(
        config=recognition_config, audio=audio
    ).result(timeout=300)

    # 각 결과(result)는 순차적으로 구분되어 있음.
    # 단, 마지막 결과에 있는 단어 리스트(alternatives[0].words)는 전체 단어를 포함하므로,
    # 모든 단어와 화자 정보를 얻으려면 마지막 결과만 사용하면 됨.
    result = response.results[-1]
    words_info = result.alternatives[0].words

    # 결과 출력
    for word_info in words_info:
        print(f"word: '{word_info.word}', speaker_tag: {word_info.speaker_tag}")
    print(True)